In [6]:
import pandas as pd
import numpy as np

In [7]:
from datasets import load_dataset

ds = load_dataset("OpenAssistant/oasst2")

In [8]:
en_ds = ds["train"].filter(lambda x: x["lang"] == "en")

In [9]:
df = en_ds.to_pandas()

In [10]:
def get_quality(labels):
    if labels and 'name' in labels:
        idx = np.where(labels['name'] == 'quality')[0]
        return labels['value'][idx[0]] if len(idx) > 0 else None
    return None

df['quality'] = df['labels'].apply(get_quality)

In [18]:
high_quality_df = df.loc[df['quality'] == 1].copy()

In [19]:
parent_lookup = df.set_index('message_id').to_dict('index')

def select_sample(r):
    parent = parent_lookup.get(r['parent_id'])

    if parent is None:
        return False, None

    parent_text = parent['text']

    if (
        pd.isna(parent['parent_id']) and
        parent['role'] == 'prompter' and
        len(parent_text) >= 200 and
        len(r['text']) >= 500
    ):
        return True, parent_text

    return False, None

high_quality_df.loc[:,'select_sample_result'] = (
    high_quality_df.apply(select_sample, axis=1)
)

high_quality_df.loc[:,'select_sample'] = (
    high_quality_df.select_sample_result.apply(lambda x: x[0] if x else None)
)
high_quality_df.loc[:,'parent_text'] = (
    high_quality_df.select_sample_result.apply(lambda x: x[1] if x else None)
)

high_quality_df.drop('select_sample_result', axis=1, inplace=True) 

In [26]:
final_selected = high_quality_df[high_quality_df['select_sample']].copy()

In [27]:
final_selected.text.apply(lambda x: len(x)).describe()

count     155.000000
mean     1771.509677
std       990.385065
min       503.000000
25%      1128.000000
50%      1600.000000
75%      2311.000000
max      9033.000000
Name: text, dtype: float64

In [29]:
final_selected.rename({"parent_text": "Query", "text": "Response"}, axis=1, inplace=True)

In [30]:
final_selected = final_selected.loc[:, ['message_tree_id', 'message_id', 'Query', 'Response', 'quality', 'labels']].reset_index(drop=True)

In [31]:
final_selected.drop_duplicates(['Query'],inplace=True)

In [32]:
final_selected.to_csv("oasst2_processed.csv", index=False)